In [1]:
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=1000, n_features=20, n_classes=3, n_informative=10, random_state=42)
df = pd.DataFrame(X, columns=[f"feat_{i}" for i in range(X.shape[1])])
df["target"] = y

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
print(f"Train: {train_df.shape}, Test: {test_df.shape}")

Train: (800, 21), Test: (200, 21)


In [2]:
import numpy as np
from autogluon.core.models import AbstractModel
from sklearn.linear_model import RidgeClassifierCV


class RidgeClassifierCVModel(AbstractModel):
    def _fit(self, X, y, **kwargs):
        self.model = RidgeClassifierCV()
        self.model.fit(X, y)

    def _predict_proba(self, X, **kwargs):
        scores = self.model.decision_function(X)
        # softmax over decision scores to get probabilities
        scores -= scores.max(axis=1, keepdims=True)
        exp_scores = np.exp(scores)
        return exp_scores / exp_scores.sum(axis=1, keepdims=True)

In [3]:
from autogluon.tabular import TabularPredictor

predictor = TabularPredictor(label="target", eval_metric="accuracy", verbosity=2)
predictor.fit(
    train_data=train_df,
    hyperparameters={
        "GBM": {},
        "RF": {},
        "XT": {},
        RidgeClassifierCVModel: {},
    },
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260602_145958"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.13.2
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #117-Ubuntu SMP PREEMPT_DYNAMIC Tue May  5 19:26:24 UTC 2026
CPU Count:          48
Pytorch Version:    2.12.0+cpu
CUDA Version:       CUDA is not available
Memory Avail:       885.92 GB / 1007.77 GB (87.9%)
Disk Space Avail:   346.39 GB / 1875.49 GB (18.5%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : New in v1.5: The state-of-the-art for tabular data. Massively better than 'best' on datasets <100000 samples by using new Tabular Foundation Models (TFMs) meta

In [5]:
predictor.leaderboard(test_df)

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,RandomForest,0.845,0.73125,accuracy,0.139753,0.079786,1.436115,0.139753,0.079786,1.436115,1,True,2
1,LightGBM,0.840,0.79375,accuracy,0.019069,0.002127,2.728600,0.019069,0.002127,2.728600,1,True,1
2,WeightedEnsemble_L2,0.840,0.79375,accuracy,0.023194,0.003142,2.789674,0.004125,0.001014,0.061074,2,True,5
3,ExtraTrees,0.830,0.74375,accuracy,0.165221,0.095639,1.135080,0.165221,0.095639,1.135080,1,True,3
4,RidgeClassifierCVModel,0.760,0.63125,accuracy,0.007335,0.001322,0.018636,0.007335,0.001322,0.018636,1,True,4
